In [1]:
import json, os, re
import pandas as pd
import sqlite3
import shutil
import numpy as np
from datetime import datetime

# Parsing Google Keep Notes into Workout JSON files

In [18]:
WEIGHT_UNITS = ['kg', 'lbs', 'bar', 'caret', 'BW']
SUPERSET_DELIM = '/'
DROPSET_DELIM = '>'
RESTPAUSE_DELIMS = ['no rest', '...', '..']
HAND_REP_DELIM = '~'

In [2]:
# Initialize data path
path = 'Data'
data_path = 'Workout logs'

## Filtering the workout logs notes

In [3]:
# # Make the workout logs folder
# if not os.path.exists(data_path):
#     os.makedirs(data_path)

# # Iterate all the files if there's no content in workout logs folder
# if len(os.listdir(data_path)) == 0:
#     for filepath in os.listdir(path):
#         # Read the file
#         full_filepath = os.path.join(path, filepath)
#         file = open(full_filepath).read()
#         # Load file as json
#         json_file = json.loads(file)

#         # Check if file is a Workout Log
#         label = 'Workout Log'
#         if 'labels' in json_file and {'name': label} in json_file['labels']:
#             shutil.copy(full_filepath, data_path)
#             print(f'Copied {filepath} to {data_path}')

## Renaming workout logs notes

In [4]:
# # Iterate all the files and check the title if it has a year
# for filepath in os.listdir(data_path):
#     full_filepath = os.path.join(data_path, filepath)
#     with open(full_filepath) as f:
#         json_file = json.load(f)

#     # Get the title
#     if 'title' in json_file:
#         try:
#             # Get year
#             pd.Timestamp(json_file['title']).year
#             print(f'Skipped renaming {filepath}')
#         except:
#             # Get the creation timestamp
#             if 'createdTimestampUsec' in json_file:
#                 timestamp = json_file['createdTimestampUsec']
#                 # Convert the timestamp to a BB dd, YYYY
#                 new_title = datetime.fromtimestamp(int(timestamp) / 1_000_000).strftime('%B %d, %Y')

#                 # Auto-increment suffix if file already exists
#                 new_filename = f'{new_title}.json'
#                 new_filepath = os.path.join(data_path, new_filename)
#                 counter = 1
#                 while os.path.exists(new_filepath):
#                     new_filename = f'{new_title}({counter}).json'
#                     new_filepath = os.path.join(data_path, new_filename)
#                     counter += 1

#                 # Rename the path
#                 os.rename(full_filepath, new_filepath)
#                 print(f'Renamed: {filepath} -> {new_filename}')

In [5]:
pd.Timestamp('Feb 13, 2020').year

2020

In [6]:
datetime.fromtimestamp(1739880806652000/1_000_000).strftime('%B %d, %Y')

'February 18, 2025'

## Making the parser

In [3]:
from utils.parser import *

### Trial 1

In [4]:
# Read the file 
with open(os.path.join(path, 'Feb 18, 2025.json')) as f:
    file = f.read()
 
# Load as json
json_file = json.loads(file)
json_file

{'color': 'DEFAULT',
 'isTrashed': False,
 'isPinned': False,
 'isArchived': False,
 'textContent': 'Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2\nCable fly (lower) - 22 lbs 12 , 33 lbs 10, 33 lbs 10\nInclined dumbbell fly - 15 lbs 10, 10, 20 lbs 10\nPec deck fly (lower) - 60 lbs 10, 70 lbs>60 lbs 7>6, 6>7\nArnold machine - 5 lbs 8, 6, \nTricep dip machine - 2x35 lbs 10, 2x(35+2.5 lbs) 10, 2x35 lbs + 4x2.5 lbs 10\nChest machine - 30 lbs 10, 45 lbs>30 lbs 2 then 8>10, 7>5\nTricep pulldown - 55 lbs>44 lbs 7F>5F, 7F>4F, 7F>3F\n\n\nWeight - 76 kg\n\n',
 'title': 'Feb 18, 2025',
 'userEditedTimestampUsec': 1739880806652000,
 'createdTimestampUsec': 1739875379471000,
 'textContentHtml': '<p dir="ltr" style="line-height:1.38;margin-top:0.0pt;margin-bottom:0.0pt;"><span style="font-size:7.2pt;font-family:\'Google Sans\';color:#000000;background-color:transparent;font-weight:400;font-style:normal;font-variant:normal;text-decoration:none

In [5]:
movements = '12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2'

In [6]:
parse_movement(movements)

['12.5 lbs 10',
 '15 lbs 10',
 '17.5 lbs 7 no rest 15 4',
 '20 lbs 3 no rest 17.5 1 no rest 15 lbs 2']

In [7]:
separate_load_from_rep('10')

Looking for kg
Looking for lbs
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is None at -1


(None, '10')

In [8]:
parse_set('12.5 lbs 10...5...2')

Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
12.5 lbs 10...5...2 is a unit set!


{'kind': 'unit', 'subsets': {'load': '12.5 lbs', 'reps': [10, 5, 2]}}

In [9]:
# text = movements

# # Initialize index and units of weight
# weight_index = 0
# weight_units = ['kg', 'lbs', 'bar', 'caret', 'BW']
# # Initialize that unit can be found
# weight_unit_found = [True, True, True, True, True]

# while any(weight_unit_found):
#     for idx, weight_unit in enumerate(weight_units):
#         # Stop looking for weight units that cannot be found
#         if not weight_unit_found[idx]:
#             continue

#         print(f'Looking for {weight_unit}')
#         # Find the weight unit
#         found_index = text.find(weight_unit)

#         # Update weight index if it's the rightmost weight unit
#         if found_index > weight_index:
#             print(f'Found {weight_unit} at {found_index}')
#             weight_index = found_index

#         if found_index == -1:
#             print(f'{weight_unit} cannot be found')
#             weight_unit_found[idx] = False

In [10]:
# # Initialize index and units of weight
# weight_index = 0
# weight_units = ['kg', 'lbs', 'bar', 'caret', 'BW']

# for idx, weight_unit in enumerate(weight_units):
#         print(f'Looking for {weight_unit}')
#         # Find the weight unit
#         found_index = text.rfind(weight_unit)

#         # Update weight index if it's the rightmost weight unit
#         if found_index > weight_index:
#             print(f'Found {weight_unit} at {found_index}')
#             weight_index = found_index

# print(f'Found rightmost weight unit at {weight_index}')

In [11]:
parse_exercise('Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2')

Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
12.5 lbs 10 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 3
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 3
15 lbs 10 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
17.5 lbs 7 no rest 15 4 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 35
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 35
20 lbs 3 no rest 17.5 1 no rest 15 lbs 2 is a unit set!


{'id': 'bench_press',
 'movements': [{'name': 'Bench press',
   'sets': [{'kind': 'unit', 'subsets': {'load': '12.5 lbs', 'reps': 10}},
    {'kind': 'unit', 'subsets': {'load': '15 lbs', 'reps': 10}},
    {'kind': 'unit', 'subsets': {'load': '17.5 lbs', 'reps': [7, '15 4']}},
    {'kind': 'unit',
     'subsets': {'load': '20 lbs 3 no rest 17.5 1 no rest 15 lbs',
      'reps': 2}}]}]}

In [12]:
group_log_by_content(json_file['textContent'])

{'exercises': ['Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2',
  'Cable fly (lower) - 22 lbs 12 , 33 lbs 10, 33 lbs 10',
  'Inclined dumbbell fly - 15 lbs 10, 10, 20 lbs 10',
  'Pec deck fly (lower) - 60 lbs 10, 70 lbs>60 lbs 7>6, 6>7',
  'Arnold machine - 5 lbs 8, 6, ',
  'Tricep dip machine - 2x35 lbs 10, 2x(35+2.5 lbs) 10, 2x35 lbs + 4x2.5 lbs 10',
  'Chest machine - 30 lbs 10, 45 lbs>30 lbs 2 then 8>10, 7>5',
  'Tricep pulldown - 55 lbs>44 lbs 7F>5F, 7F>4F, 7F>3F'],
 'notes': [],
 'comments': [],
 'weight': ['76 kg']}

In [13]:
parse_workout_log(json_file['textContent'])

Parsing exercise: Bench press - 12.5 lbs 10, 15 lbs 10, 17.5 lbs 7 no rest 15 4, 20 lbs 3 no rest 17.5 1 no rest 15 lbs 2
Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
12.5 lbs 10 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 3
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 3
15 lbs 10 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 5
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 5
17.5 lbs 7 no rest 15 4 is a unit set!
Looking for kg
Looking for lbs
Found lbs at 35
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 35
20 lbs 3 no rest 17.5 1 no rest 15 lbs 2 is a unit set!
Parsing exercise: Cable fly (lower) - 22 lbs 12 , 33 lbs 10, 33 lbs 10
Looking for kg
Looking for lbs
Found lbs at 3
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 3
22 lbs 12  is

{'exercises': [{'id': 'bench_press',
   'movements': [{'name': 'Bench press',
     'sets': [{'kind': 'unit', 'subsets': {'load': '12.5 lbs', 'reps': 10}},
      {'kind': 'unit', 'subsets': {'load': '15 lbs', 'reps': 10}},
      {'kind': 'unit', 'subsets': {'load': '17.5 lbs', 'reps': [7, '15 4']}},
      {'kind': 'unit',
       'subsets': {'load': '20 lbs 3 no rest 17.5 1 no rest 15 lbs',
        'reps': 2}}]}]},
  {'id': 'cable_fly_(lower)',
   'movements': [{'name': 'Cable fly (lower)',
     'sets': [{'kind': 'unit', 'subsets': {'load': '22 lbs', 'reps': 12}},
      {'kind': 'unit', 'subsets': {'load': '33 lbs', 'reps': 10}},
      {'kind': 'unit', 'subsets': {'load': '33 lbs', 'reps': 10}}]}]},
  {'id': 'inclined_dumbbell_fly',
   'movements': [{'name': 'Inclined dumbbell fly',
     'sets': [{'kind': 'unit', 'subsets': {'load': '15 lbs', 'reps': 10}},
      {'kind': 'unit', 'subsets': {'load': None, 'reps': 10}},
      {'kind': 'unit', 'subsets': {'load': '20 lbs', 'reps': 10}}]}]},

## Test

In [14]:
workout_log = """
Lat pulldown - 45 kg 10, 50 kg 10, 55 kg > 50 kg 5 NROM > 4F NROM, 45 kg 5 slow... 5 fast... 1 hold 5 sec
Reverse delt fly - 66 lbs 10, 77 lbs 10, 88 lbs > 77 lbs 5 > 5
Chest-supported rows - 2x(15kg+ 5 kg) 12, 2x(15 kg + 5 kg + 2.5kg) 10*, 2x(15 kg + 5 kg + 2x2.5kg) > 2x(15 kg + 5 kg + 2.5kg) 6...2 NROM>2
Db shrugs / Lat raise - 30 lbs / 20 lbs 15/8, 15/7...3, 15/10
Db shoulder press - 20 lbs 10, 15, 15
Back machine - 20 kg 8**~8, 5...3 NROM~5...3 NROM, 5...3 NROM~5...3 NROM
Barbell rows - bar+ 2x(10 kg + 5 kg) 10, bar + 2x(10kg + 2x5kg) 10, bar+2x(10kg+2x5kg+2.5kg) 10
Preacher curl - bar+2x5kg 10, 9F, 7F
Face pull - 35 kg 12, 40 kg 10, 45 kg 10
Forearm curl / Reverse forearm curl - 15 lbs / 10 lbs 15/10, 15/10, 15/10
Bicep curl - 20 lbs 6~7...2~2, 5~5...2~2, 20 lbs > 15 lbs 5~5...1~1 > 2~2...1~1

Notes:
* - struggled a bit by 8th rep
** - sensible fatigue

Weight: 
"""

In [15]:
parse_workout_log(workout_log)

Parsing exercise: Lat pulldown - 45 kg 10, 50 kg 10, 55 kg > 50 kg 5 NROM > 4F NROM, 45 kg 5 slow... 5 fast... 1 hold 5 sec
Looking for kg
Found kg at 3
Looking for lbs
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is kg at 3
45 kg 10 is a unit set!
Looking for kg
Found kg at 3
Looking for lbs
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is kg at 3
50 kg 10 is a unit set!
Looking for kg
Found kg at 11
Looking for lbs
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is kg at 11
55 kg > 50 kg 5 NROM > 4F NROM is a drop set!
Looking for kg
Found kg at 3
Looking for lbs
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is kg at 3
45 kg 5 slow... 5 fast... 1 hold 5 sec is a unit set!
Parsing exercise: Reverse delt fly - 66 lbs 10, 77 lbs 10, 88 lbs > 77 lbs 5 > 5
Looking for kg
Looking for lbs
Found lbs at 3
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 3
66 lbs 1

ValueError: too many values to unpack (expected 2)

## Superset treatment

In [8]:
def separate_name_from_movements(text):
    # Get the name by splitting by hyphen
    name, movements = text.split(' - ', 1)
    # Clean name by whitespace
    name = name.strip()

    return name, movements

In [ ]:
superset_movement = 'Db shrugs / Lat raise - 30 lbs / 15 lbs 15/12, 35 lbs / 20 lbs 15/10'

In [14]:
superset_name, superset_movements = separate_name_from_movements(superset_movement)

In [13]:
def get_superset_exercises_amount(name):
    return len(name.split('/'))

In [ ]:
movement_list = re.split(r',\s*', superset_movements)
for movement in movement_list:
    load, rep = separate_load_from_rep(movement)

['30 lbs / 15 lbs 15/12', '35 lbs / 20 lbs 15/10']

In [12]:
separate_load_from_rep('30 lbs / 15 lbs 15/12')

Looking for kg
Looking for lbs
Found lbs at 12
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 12


('30 lbs / 15 lbs', '15/12')

In [27]:
def convert_superset_to_sets(text):
    # Separate name from movements
    name, movements = separate_name_from_movements(text)

    # Split name by delimiter
    name_list = [n.strip() for n in name.split(SUPERSET_DELIM)]
    # Split movements by comma
    movement_list = re.split(r',\s*', movements)

    # Initialize result list
    result_list = [f'{name} -' for name in name_list]
    # Iterate over each movement
    for movement in movement_list:
        # Separate load from reps
        load, rep = separate_load_from_rep(movement)
        # Separate the load and rep by superset delimiter
        load_list = [l.strip() for l in load.split(SUPERSET_DELIM)]
        rep_list = [r.strip() for r in rep.split(SUPERSET_DELIM)]

        # Append the load and rep to the corresponding exercise string in the result list
        for idx, (l, r) in enumerate(zip(load_list, rep_list)):
            result_list[idx] += f' {l} {r},'

    # Clean trailing commas
    result_list = [s.rstrip(',') for s in result_list]

    return result_list

In [28]:
convert_superset_to_sets(superset_movement)

Looking for kg
Looking for lbs
Found lbs at 12
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 12
Looking for kg
Looking for lbs
Found lbs at 12
Looking for bar
Looking for caret
Looking for BW
Rightmost weight unit is lbs at 12


['Db shrugs - 30 lbs 15, 35 lbs 15', 'Lat raise - 15 lbs 12, 20 lbs 10']